# ReAct: 추론과 행동을 결합한 프롬프팅

ReAct(Reasoning + Acting)는 언어 모델이 추론(Reasoning)과 행동(Acting)을 번갈아가며 복잡한 작업을 수행하도록 하는 프롬프팅 기법입니다. 이 방식은 모델이 단계적으로 생각하고, 필요한 정보를 수집하며, 행동을 취하는 과정을 통해 더 정확하고 신뢰할 수 있는 결과를 얻을 수 있게 합니다.

이 노트북에서는 ReAct 기법의 기본 개념을 배우고 다양한 작업에 적용해보겠습니다.

In [1]:
# 필요한 라이브러리 및 모듈 임포트
import os
import sys
import json
import re
from dotenv import load_dotenv

# utils 디렉토리의 helpers.py 모듈을 임포트하기 위한 경로 설정
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.helpers import get_completion

# .env 파일 로드
load_dotenv()

True

## 1. ReAct의 기본 개념

ReAct는 다음 세 가지 주요 요소로 구성됩니다:

1. **사고(Thought)**: 모델이 현재 상황을 분석하고 다음에 취할 행동을 계획하는 단계
2. **행동(Action)**: 모델이 외부 환경(도구, API 등)과 상호작용하는 단계
3. **관찰(Observation)**: 행동의 결과를 관찰하고 다음 사고 단계로 피드백하는 단계

이러한 과정을 반복하며 모델은 복잡한 문제를 단계적으로 해결해 나갑니다. ReAct 방식의 주요 장점은 다음과 같습니다:

- 모델의 추론 과정을 명시적으로 볼 수 있어 투명성 증가
- 외부 도구와의 상호작용을 통한 정확도 향상
- 복잡한 다단계 작업을 체계적으로 수행 가능
- 모델의 환각(hallucination) 감소

In [2]:
from utils.react import create_react_prompt, serialize_tools

def react_agent(query, tools, max_iterations=5, verbose=False):
    tools_dict = {tool.name: tool for tool in tools}
    tools_json = serialize_tools(tools)
    
    iterations = 0
    conversation_history = []
    
    while iterations < max_iterations:
        # 프롬프트 생성 
        prompt = create_react_prompt(query, tools_json)
        if conversation_history:
            prompt += "\n\n대화 기록:\n" + "\n".join(conversation_history)
        
        # LLM 응답 요청
        response = get_completion(prompt)
        
        if verbose:
            print(f"프롬프트:\n{prompt}\n")
            print(f"응답:\n{response}\n")
        
        try:
            # JSON 응답 파싱
            response_data = json.loads(response)
            
            # 사고 과정 기록
            thoughts = response_data.get('thoughts', '')
            conversation_history.append(f"사고: {thoughts}")
            
            # 도구 사용 또는 직접 답변
            if response_data.get('use_tool', False):
                tool_name = response_data.get('tool_name')
                tool_params = response_data.get('tool_params', {})
                
                if tool_name in tools_dict:
                    # 도구 실행
                    result = tools_dict[tool_name].execute(tool_params)
                    conversation_history.append(f"도구 사용: {tool_name}, 결과: {result}")
                    
                    # 다음 반복을 위해 결과 추가
                    query = f"이전 질문: {query}\n도구 결과: {result}"
                else:
                    error_msg = f"오류: 존재하지 않는 도구 {tool_name}"
                    conversation_history.append(error_msg)
                    return error_msg
            else:
                # 최종 답변 반환
                return response_data.get('answer', '답변을 찾을 수 없습니다.')
        
        except json.JSONDecodeError:
            error_msg = "오류: 응답을 JSON으로 파싱할 수 없습니다."
            conversation_history.append(error_msg)
            return error_msg
        
        iterations += 1
    
    return "최대 반복 횟수에 도달했습니다."

## 2. 계산기 도구를 사용하는 예제
- 계산기 Tool을 등록하고 사용하는 예제입니다.

In [3]:
import math
from utils.react import Tool

class CalculatorTool(Tool):
    def __init__(self):
        super().__init__(
            name="calculator",
            description="사칙연산을 수행할 수 있는 계산기 도구입니다.",
            params_schema={"expression": "계산할 수학 표현식"}
        )
    
    def execute(self, params):
        try:
            expression = params.get("expression", "")
            allowed_chars = set('0123456789.+-*/() ')
            if not all(c in allowed_chars for c in expression):
                return "오류: 지원되지 않는 연산자나 함수가 포함되어 있습니다."
            result = eval(expression)
            return result
        except Exception as e:
            return f"계산 오류: {str(e)}"

class SQRTCalculatorTool(Tool):
    def __init__(self):
        super().__init__(
            name="sqrt_calculator",
            description="제곱근을 계산할 수 있는 도구입니다.",
            params_schema={"number": "제곱근을 계산할 수"}
        )
    
    def execute(self, params):
        try:
            number = params.get("number", "")
            result = math.sqrt(number)
            return result
        except Exception as e:
            return f"제곱근 오류: {str(e)}"

In [4]:
from utils.react import serialize_tools

tools = [CalculatorTool(), SQRTCalculatorTool()]
print(f"Serialized tools:\n{serialize_tools(tools)}\n\n")

query = "345의 제곱근에 2.5를 더한 값은 얼마인가요?"
result = react_agent(query, tools, verbose=True)
print(f"질문: {query}")
print(f"최종 답변: {result}")

Serialized tools:
[{"name": "calculator", "description": "사칙연산을 수행할 수 있는 계산기 도구입니다.", "params": {"expression": "계산할 수학 표현식"}}, {"name": "sqrt_calculator", "description": "제곱근을 계산할 수 있는 도구입니다.", "params": {"number": "제곱근을 계산할 수"}}]


프롬프트:

    당신은 사용자 질문에 답변하는 AI 어시스턴트입니다. 다음 도구들을 사용할 수 있습니다:
    
    [{"name": "calculator", "description": "사칙연산을 수행할 수 있는 계산기 도구입니다.", "params": {"expression": "계산할 수학 표현식"}}, {"name": "sqrt_calculator", "description": "제곱근을 계산할 수 있는 도구입니다.", "params": {"number": "제곱근을 계산할 수"}}]
    
    사용자의 질문에 답하기 위해 다음 형식으로 응답하세요:
    {
        "thoughts": "문제 해결을 위한 사고 과정",
        "use_tool": true|false,
        "tool_name": "사용할 도구 이름", (use_tool이 true인 경우에만)
        "tool_params": {도구에 필요한 매개변수}, (use_tool이 true인 경우에만)
        "answer": "사용자 질문에 대한 최종 답변" (use_tool이 false인 경우에만)
    }
    
    응답 규칙:
    - 구조화된 JSON 형식으로만 응답해야 합니다
    - ```json 등 Wrapping을 사용하지 말아야 합니다
    - 복잡한 문제는 여러 단계로 나누어 해결하세요
    - 도구를 사용할 때는 정확한 매개변수를 제공하세요
    
    사용자 질문: 345의 제곱근에 2.5를

## 3. 데이터 분석 도구 사용 예제

In [5]:
from utils.react import Tool
import numpy as np

class DataAnalyzerTool(Tool):
    def __init__(self):
        super().__init__(
            name="data_analyzer",
            description="데이터셋에 대한 기본 통계 분석을 수행합니다",
            params_schema={
                "data": "분석할 쉼표로 구분된 숫자 리스트",
                "operation": "수행할 연산 ('mean', 'median', 'std', 'min', 'max', 'count' 중 하나)"
            }
        )
    
    def execute(self, params):
        try:
            data_str = params.get("data", "")
            operation = params.get("operation", "").lower()
            
            # 문자열을 숫자 리스트로 변환
            data = [float(x.strip()) for x in data_str.split(",") if x.strip()]
            
            if not data:
                return "오류: 유효한 데이터가 없습니다."
            
            # 통계 연산 수행
            if operation == "mean":
                return np.mean(data)
            elif operation == "median":
                return np.median(data)
            elif operation == "std":
                return np.std(data)
            elif operation == "min":
                return min(data)
            elif operation == "max":
                return max(data)
            elif operation == "count":
                return len(data)
            else:
                return f"오류: 지원되지 않는 연산 '{operation}'. 지원되는 연산: mean, median, std, min, max, count"
        
        except Exception as e:
            return f"분석 오류: {str(e)}"

In [6]:
# 데이터 분석 도구 인스턴스 생성
tools = [
    DataAnalyzerTool()
]
print(f"Serialized tools:\n{serialize_tools(tools)}\n\n")

# 데이터 분석 도구를 사용한 ReAct 에이전트 실행
query = "다음 데이터셋의 평균과 최댓값을 알려주세요: 23, 45, 67, 12, 89, 54, 32, 78"    
result = react_agent(query, tools, verbose=True)
print(f"질문: {query}")
print(f"최종 답변: {result}")

Serialized tools:
[{"name": "data_analyzer", "description": "데이터셋에 대한 기본 통계 분석을 수행합니다", "params": {"data": "분석할 쉼표로 구분된 숫자 리스트", "operation": "수행할 연산 ('mean', 'median', 'std', 'min', 'max', 'count' 중 하나)"}}]


프롬프트:

    당신은 사용자 질문에 답변하는 AI 어시스턴트입니다. 다음 도구들을 사용할 수 있습니다:
    
    [{"name": "data_analyzer", "description": "데이터셋에 대한 기본 통계 분석을 수행합니다", "params": {"data": "분석할 쉼표로 구분된 숫자 리스트", "operation": "수행할 연산 ('mean', 'median', 'std', 'min', 'max', 'count' 중 하나)"}}]
    
    사용자의 질문에 답하기 위해 다음 형식으로 응답하세요:
    {
        "thoughts": "문제 해결을 위한 사고 과정",
        "use_tool": true|false,
        "tool_name": "사용할 도구 이름", (use_tool이 true인 경우에만)
        "tool_params": {도구에 필요한 매개변수}, (use_tool이 true인 경우에만)
        "answer": "사용자 질문에 대한 최종 답변" (use_tool이 false인 경우에만)
    }
    
    응답 규칙:
    - 구조화된 JSON 형식으로만 응답해야 합니다
    - ```json 등 Wrapping을 사용하지 말아야 합니다
    - 복잡한 문제는 여러 단계로 나누어 해결하세요
    - 도구를 사용할 때는 정확한 매개변수를 제공하세요
    
    사용자 질문: 다음 데이터셋의 평균과 최댓값을 알려주세요: 23, 45, 67, 12, 89, 54, 32, 78
    

## 4. 저장소를 활용한 일정 관리 도구 예제
- 본 예제에서는 저장소 대신 InMemory Map에 데이터를 저장하도록 구현되어있습니다.

In [7]:
class CalendarSystem:
    def __init__(self):
        # 간단히 메모리에 일정 저장 (실제로는 DB 사용)
        self._events = {}
        self._event_id_counter = 1
    
    def add_event(self, title, date, time, description=""):
        """일정 추가"""
        event_id = self._event_id_counter
        self._events[event_id] = {
            "id": event_id,
            "title": title,
            "date": date,
            "time": time,
            "description": description
        }
        self._event_id_counter += 1
        return event_id
    
    def get_events(self, date=None):
        """일정 조회"""
        if date:
            return [event for event in self._events.values() if event["date"] == date]
        return list(self._events.values())
    
    def delete_event(self, event_id):
        """일정 삭제"""
        if event_id in self._events:
            deleted_event = self._events.pop(event_id)
            return f"일정 '{deleted_event['title']}'이(가) 삭제되었습니다."
        return f"ID {event_id}에 해당하는 일정을 찾을 수 없습니다."

In [8]:
from utils.react import Tool

# 공유 캘린더 시스템 인스턴스 생성
calendar_system = CalendarSystem()

class AddEventTool(Tool):
    def __init__(self):
        super().__init__(
            name="add_event",
            description="새 일정을 추가합니다",
            params_schema={
                "title": "일정 제목",
                "date": "일정 날짜 (YYYY-MM-DD 형식)",
                "time": "일정 시간 (HH:MM 형식)",
                "description": "일정 설명 (선택사항)"
            }
        )
    
    def execute(self, params):
        title = params.get("title")
        date = params.get("date")
        time = params.get("time")
        description = params.get("description", "")
        
        # 파라미터 유효성 검사
        if not title or not date or not time:
            return "오류: 제목, 날짜, 시간은 필수 항목입니다."
        
        # 날짜 형식 검사 (간단한 검사)
        if not re.match(r"\d{4}-\d{2}-\d{2}", date):
            return "오류: 날짜는 YYYY-MM-DD 형식이어야 합니다."
        
        # 시간 형식 검사
        if not re.match(r"\d{2}:\d{2}", time):
            return "오류: 시간은 HH:MM 형식이어야 합니다."
        
        # 일정 추가
        event_id = calendar_system.add_event(title, date, time, description)
        return f"일정이 추가되었습니다 (ID: {event_id})"

class GetEventsTool(Tool):
    def __init__(self):
        super().__init__(
            name="get_events",
            description="특정 날짜 또는 모든 일정을 조회합니다",
            params_schema={
                "date": "조회할 날짜 (YYYY-MM-DD 형식). 비워두면 모든 일정 조회"
            }
        )
    
    def execute(self, params):
        date = params.get("date", "")
        
        # 날짜가 지정된 경우 형식 검사
        if date and not re.match(r"\d{4}-\d{2}-\d{2}", date):
            return "오류: 날짜는 YYYY-MM-DD 형식이어야 합니다."
        
        # 일정 조회
        events = calendar_system.get_events(date if date else None)
        
        # 결과 포맷팅
        if not events:
            return "일정이 없습니다."
        
        # 간단한 테이블 형식으로 출력
        result = "일정 목록:\n"
        for event in events:
            result += f"ID: {event['id']} | {event['date']} {event['time']} | {event['title']}"
            if event['description']:
                result += f" | {event['description']}"
            result += "\n"
        
        return result

class DeleteEventTool(Tool):
    def __init__(self):
        super().__init__(
            name="delete_event",
            description="ID로 일정을 삭제합니다",
            params_schema={
                "event_id": "삭제할 일정의 ID"
            }
        )
    
    def execute(self, params):
        event_id_str = params.get("event_id", "")
        
        try:
            event_id = int(event_id_str)
        except ValueError:
            return "오류: 일정 ID는 숫자여야 합니다."
        
        # 일정 삭제
        return calendar_system.delete_event(event_id)

In [9]:
from utils.react import serialize_tools

# 일정 관리 도구 인스턴스 생성
tools = [
    AddEventTool(),
    GetEventsTool(),
    DeleteEventTool()
]
print(f"Serialized tools:\n{serialize_tools(tools)}\n\n")

    
# 1. 일정 추가 예제
query1 = "내일 오후 2시에 팀 회의 일정을 추가해주세요. 오늘 날짜는 2025-04-01입니다."
print(f"질문 1: {query1}")
result1 = react_agent(query1, tools, verbose=False)
print(f"최종 답변 1: {result1}\n")

# 2. 일정 조회 예제
query2 = "이번 주 일정을 모두 보여주세요"
print(f"질문 2: {query2}")
result2 = react_agent(query2, tools, verbose=False)
print(f"최종 답변 2: {result2}\n")
    
# 3. 일정 삭제 예제
query3 = "팀 회의 일정을 삭제해주세요"
print(f"질문 3: {query3}")
result3 = react_agent(query3, tools, verbose=False)
print(f"최종 답변 3: {result3}\n")

# 4. 일정 조회 예제
query4 = "이번 주 일정을 모두 보여주세요"
print(f"질문 4: {query4}")
result4 = react_agent(query4, tools, verbose=False)
print(f"최종 답변 4: {result4}\n")


Serialized tools:
[{"name": "add_event", "description": "새 일정을 추가합니다", "params": {"title": "일정 제목", "date": "일정 날짜 (YYYY-MM-DD 형식)", "time": "일정 시간 (HH:MM 형식)", "description": "일정 설명 (선택사항)"}}, {"name": "get_events", "description": "특정 날짜 또는 모든 일정을 조회합니다", "params": {"date": "조회할 날짜 (YYYY-MM-DD 형식). 비워두면 모든 일정 조회"}}, {"name": "delete_event", "description": "ID로 일정을 삭제합니다", "params": {"event_id": "삭제할 일정의 ID"}}]


질문 1: 내일 오후 2시에 팀 회의 일정을 추가해주세요. 오늘 날짜는 2025-04-01입니다.
최종 답변 1: 일정이 성공적으로 추가되었습니다. 추가적인 도움이 필요하시면 말씀해주세요.

질문 2: 이번 주 일정을 모두 보여주세요
최종 답변 2: 이번 주의 일정은 다음과 같습니다:
- 2025-04-02 14:00 | 팀 회의

질문 3: 팀 회의 일정을 삭제해주세요
최종 답변 3: 팀 회의 일정이 이미 삭제되었습니다.

질문 4: 이번 주 일정을 모두 보여주세요
최종 답변 4: 이번 주에 등록된 일정이 없습니다.



## 5. OpenAI API에 내장된 Tools 예제
- 최근 LLM API에는 도구를 사용하는 기능이 내장되어있습니다. 이를 활용하면 복잡한 로직 구현없이 쉽게 도구를 등록하고 실행할 수 있습니다.

In [13]:
from utils.helpers import client
import json
import os

# 계산기 도구 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "간단한 사칙연산을 수행합니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 표현식 (예: '2 + 2', '3 * 4')"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

def calculate(expression):
    try:
        # 안전한 표현식만 허용
        allowed_chars = set('0123456789.+-*/() ')
        if not all(c in allowed_chars for c in expression):
            return "오류: 지원되지 않는 연산자가 포함되어 있습니다."
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {str(e)}"

def process_calculation(query):
    # API 호출
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": query}],
        tools=tools,
        tool_choice="auto"
    )
    
    message = response.choices[0].message
    
    # 도구 호출이 있는 경우
    if hasattr(message, "tool_calls") and message.tool_calls:
        tool_call = message.tool_calls[0]  # 첫 번째 도구 호출 사용
        function_args = json.loads(tool_call.function.arguments)
        print(f"도구 사용: {tool_call.function.name}, 파라미터: {function_args}")
        
        # 계산 실행
        result = calculate(function_args["expression"])
        
        # 결과를 포함하여 최종 응답 생성
        final_response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "user", "content": query},
                {"role": "assistant", "content": None, "tool_calls": [tool_call]},
                {"role": "tool", "tool_call_id": tool_call.id, "content": result}
            ]
        )
        
        return final_response.choices[0].message.content
    
    return message.content

# 실행 예제
queries = [
    "23 곱하기 45는 얼마인가요?",
    "128 나누기 4의 결과를 알려주세요",
    "17 더하기 25 빼기 10은?"
]
    
for query in queries:
    print(f"\n질문: {query}")
    response = process_calculation(query)
    print(f"답변: {response}")



질문: 23 곱하기 45는 얼마인가요?
도구 사용: calculator, 파라미터: {'expression': '23 * 45'}
답변: 23 곱하기 45는 1035입니다.

질문: 128 나누기 4의 결과를 알려주세요
도구 사용: calculator, 파라미터: {'expression': '128 / 4'}
답변: 128 나누기 4의 결과는 32입니다.

질문: 17 더하기 25 빼기 10은?
도구 사용: calculator, 파라미터: {'expression': '17 + 25 - 10'}
답변: 32입니다.


## 5. ReAct의 한계 및 개선 방안

In [10]:
# ReAct의 한계 및 개선 방안에 대한 정보 제공
react_limitations_prompt = """
ReAct(Reasoning and Acting) 방식의 한계와 이를 개선하기 위한 방안을 자세히 설명해주세요.
다음 측면에서 논의해주세요:
1. 복잡한 추론이 필요한 경우의 한계
2. 도구 오용 및 오류 처리
3. 무한 루프 및 효율성 문제
4. 개선 방안 및 최신 연구 동향
"""

limitations_response = get_completion(react_limitations_prompt)
print(limitations_response)

ReAct(Reasoning and Acting) 방식은 인공지능 에이전트가 주어진 문제를 해결하기 위해 논리적 추론과 행동을 결합하는 접근 방식입니다. 그러나 이 방식에는 여러 가지 한계가 존재하며, 이를 개선하기 위한 다양한 연구가 진행되고 있습니다. 아래에서는 ReAct 방식의 한계와 개선 방안을 제시합니다.

1. **복잡한 추론이 필요한 경우의 한계**

   - **한계**: ReAct 방식은 일반적으로 규칙 기반의 추론을 사용하며, 이는 복잡한 문제나 비구조적인 데이터를 다루기 어려울 수 있습니다. 복잡한 문제는 종종 다단계 추론과 다양한 변수를 고려해야 하므로, 단순한 규칙 기반 접근 방식으로는 한계가 발생합니다.

   - **개선 방안**: 이를 개선하기 위해 딥러닝 기반의 추론 모델이나 강화학습을 활용할 수 있습니다. 이러한 방법들은 비선형적이고 다차원적인 문제에 더 적합하며, 복잡한 패턴을 학습할 수 있습니다. 또한, 메타러닝 기법을 적용하여 에이전트가 스스로 학습하고 적응할 수 있도록 하는 연구가 진행되고 있습니다.

2. **도구 오용 및 오류 처리**

   - **한계**: ReAct 방식에서는 에이전트가 다양한 도구를 사용하여 문제를 해결하려고 하지만, 도구의 오용이나 예상치 못한 오류가 발생할 수 있습니다. 이러한 오류는 시스템의 신뢰성을 저하시킬 수 있습니다.

   - **개선 방안**: 이를 해결하기 위해 에이전트가 도구 사용 중 발생할 수 있는 오류를 예측하고 대처할 수 있는 메커니즘을 설계해야 합니다. 예를 들어, 오류 발생 시 자동으로 복구하거나 다른 접근 방식을 시도하는 기능을 포함할 수 있습니다. 또한, 에이전트가 다양한 시나리오에서 도구를 안전하게 사용할 수 있도록 시뮬레이션 기반의 훈련을 강화하는 연구도 필요합니다.

3. **무한 루프 및 효율성 문제**

   - **한계**: ReAct 방식은 복잡한 문제를 해결하는 과정에서 무한 루프에 빠지거나 비효율적으로 작동할 수 있습니다. 이는 특히 에이전트가 잘못

## 6. ReAct 패턴과 함께 사용하면 좋은 기법들

### 6.1 점진적 요약 기법
이전 히스토리를 추가하다보면 토큰수를 과도하게 사용하는 경우가 생길 수 있어요. 이 경우 히스토리중 필요한 데이터만 남기고 요약하는 식으로 최적화를 진행할 수 있어요.

In [17]:
from utils.conversation_manager import ConversationManager

# 대화 관리자와 도구 추적기 초기화
conversation = ConversationManager(max_tokens=10)
   
# 예제 1: 대화 관리
print("=== 대화 관리 예제 ===")
    
# 사용자와 어시스턴트의 대화 추가
conversation.add_message("user", "안녕하세요! 날씨 정보를 알려주세요.")
conversation.add_message("assistant", "현재 서울의 날씨는 맑고 기온은 20도입니다.")
conversation.add_message("user", "내일의 날씨는 어떨까요?")
conversation.add_message("assistant", "내일은 흐리고 비가 올 예정입니다. 기온은 18도입니다.")
    
# 현재 컨텍스트 출력
print("\n현재 대화 컨텍스트:")
for msg in conversation.get_context():
    print(f"{msg['role']}: {msg['content']}")


=== 대화 관리 예제 ===

현재 대화 컨텍스트:
system: 이전 대화 요약: 사용자가 서울의 현재 날씨를 물었고, 맑고 기온이 20도라는 정보를 받았습니다.
assistant: 내일은 흐리고 비가 올 예정입니다. 기온은 18도입니다.


### 6.2 도구 사용 내역 관리
도구를 호출하는 식으로 사용하다보면 어떤 빈도로 도구를 사용하는지 추적할 필요가 있습니다.

In [16]:
from utils.tool_tracker import ToolUsageTracker
tool_tracker = ToolUsageTracker()
 
# 날씨 조회 도구 사용 기록
tool_tracker.record_usage(
    "weather_api",
    {"city": "서울", "date": "2024-04-01"},
    {"temperature": 20, "condition": "맑음"},
    success=True
)
    
tool_tracker.record_usage(
    "weather_api",
    {"city": "서울", "date": "2024-04-02"},
    {"temperature": 18, "condition": "비"},
    success=True
)
    
# 다른 도구 사용 예제
tool_tracker.record_usage(
    "calculator",
    {"expression": "23 * 45"},
    1035,
    success=True
)

tool_tracker.record_usage(
    "translator",
    {"text": "Hello", "target_lang": "ko"},
    "안녕하세요",
    success=True
)
    
# 날씨 관련 기록 검색
print("\n날씨 관련 도구 사용 기록:")
weather_history = tool_tracker.get_relevant_history("날씨 서울")
for usage in weather_history:
    print(f"도구: {usage['tool']}")
    print(f"매개변수: {usage['params']}")
    print(f"결과: {usage['result']}\n")

# 도구 사용 통계 출력
print("도구 사용 통계:")
stats = tool_tracker.get_tool_statistics()
for tool_name, tool_stats in stats.items():
    print(f"\n도구: {tool_name}")
    print(f"총 사용 횟수: {tool_stats['total_uses']}")
    print(f"성공 횟수: {tool_stats['successful_uses']}")
    print(f"성공률: {tool_stats['success_rate']:.1f}%")


날씨 관련 도구 사용 기록:
도구: weather_api
매개변수: {'city': '서울', 'date': '2024-04-02'}
결과: {'temperature': 18, 'condition': '비'}

도구: weather_api
매개변수: {'city': '서울', 'date': '2024-04-01'}
결과: {'temperature': 20, 'condition': '맑음'}

도구 사용 통계:

도구: weather_api
총 사용 횟수: 2
성공 횟수: 2
성공률: 100.0%

도구: calculator
총 사용 횟수: 1
성공 횟수: 1
성공률: 100.0%

도구: translator
총 사용 횟수: 1
성공 횟수: 1
성공률: 100.0%
